In [1]:
import cv2
import numpy as np

# Загружаем изображение
img = cv2.imread('5table.jpg')


# 1. Преобразуем в оттенки серого и находим края
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
edges = cv2.Canny(gray, 50, 150)

# 2. Находим линии с помощью преобразования Хафа
lines = cv2.HoughLinesP(
    edges, 
    rho=1, 
    theta=np.pi/180, 
    threshold=100,
    minLineLength=100,
    maxLineGap=10
)

# 3. Рисуем найденные линии
line_img = img.copy()
corners = []

if lines is not None:
    # Собираем все линии
    line_params = []
    for line in lines:
        x1, y1, x2, y2 = line[0]
        cv2.line(line_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        # Параметры линии: Ax + By + C = 0
        A = y2 - y1
        B = x1 - x2
        C = x2*y1 - x1*y2
        line_params.append((A, B, C, x1, y1, x2, y2))
    
    # 4. Находим пересечения линий (углы)
    for i in range(len(line_params)):
        for j in range(i+1, len(line_params)):
            A1, B1, C1, _, _, _, _ = line_params[i]
            A2, B2, C2, _, _, _, _ = line_params[j]
            
            # Решаем систему уравнений для нахождения точки пересечения
            det = A1*B2 - A2*B1
            if abs(det) > 1e-6:
                x = int((B1*C2 - B2*C1) / det)
                y = int((C1*A2 - C2*A1) / det)
                
                # Добавляем угол, если он в пределах изображения
                if 0 <= x < img.shape[1] and 0 <= y < img.shape[0]:
                    corners.append((x, y))

# 5. Рисуем углы на отдельном изображении
corner_img = img.copy()
for corner in corners:
    cv2.circle(corner_img, corner, 4, (0, 0, 255), -1)


cv2.imshow('Исходное изображение', img)
cv2.imshow('Найденные линии', line_img)
cv2.imshow('Найденные углы', corner_img)


cv2.waitKey(0)
cv2.destroyAllWindows()

In [5]:
import cv2
import numpy as np


image_path = '5sharik.jpg'  
img = cv2.imread(image_path)



# Создаем копию для отображения результатов
result = img.copy()

# Преобразуем в оттенки серого
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Применяем размытие для уменьшения шума
blurred = cv2.GaussianBlur(gray, (9, 9), 2)

# Определяем размеры изображения
rows = blurred.shape[0]

# Находим окружности с помощью преобразования Хафа
circles = cv2.HoughCircles(
    blurred, 
    cv2.HOUGH_GRADIENT, 
    dp=1,           # разрешающая способность аккумулятора
    minDist=rows/8, # минимальное расстояние между центрами окружностей
    param1=100,     # верхний порог для детектора Кэнни
    param2=30,      # порог для аккумулятора (меньше -> больше окружностей)
    minRadius=10,   # минимальный радиус
    maxRadius=0     # максимальный радиус (0 = автоматически)
)

# Отображаем найденные окружности

    # Преобразуем координаты и радиусы в целые числа
circles = np.uint16(np.around(circles))



for i, circle in enumerate(circles[0, :]):
    center = (circle[0], circle[1])  # координаты центра
    radius = circle[2]  # радиус
    
    # Рисуем внешнюю окружность
    cv2.circle(result, center, radius, (0, 255, 0), 3)
    
    # Рисуем центр окружности
    cv2.circle(result, center, 2, (0, 0, 255), 3)
        
        
        
        

cv2.imshow('Исходное изображение', img)
cv2.imshow('Найденные окружности', result)


cv2.waitKey(0)
cv2.destroyAllWindows()

In [2]:
import cv2
import numpy as np

def process_ball_center_mass(path):
    # Загружаем изображение
    img = cv2.imread(path)

    # Перевод в HSV
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # Диапазон ЖЁЛТОГО шарика
    hsv_min = np.array((20, 90, 90))
    hsv_max = np.array((35, 255, 255))

    # Бинарная маска
    mask = cv2.inRange(hsv, hsv_min, hsv_max)

    # Морфологическая обработка (убираем шум и дырки)
    kernel = np.ones((7, 7), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

  
    M = cv2.moments(mask)

    cx = int(M["m10"] / M["m00"])
    cy = int(M["m01"] / M["m00"])

    center = (cx, cy)

    # Максимальное расстояние от центра до пикселя объекта
    ys, xs = np.where(mask == 255)
    distances = np.sqrt((xs - cx) ** 2 + (ys - cy) ** 2)
    r = int(np.max(distances))

   
    out = img.copy()

    # Окружность вокруг шарика
    cv2.circle(out, center, r, (0, 0, 255), 3)

    # Центр масс
    cv2.circle(out, center, 6, (255, 0, 0), -1)

    return out, mask


result2, mask = process_ball_center_mass("5sharik.jpg")

cv2.imshow("Ball via Mass Center", result2)
cv2.imshow("Ball Mask", mask)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [4]:
cv2.imshow("Ball via Mass Center", result2)
cv2.imshow('Найденные окружности', result)
cv2.waitKey(0)
cv2.destroyAllWindows()
